# Numbers needed for text

In [2]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import copy
import glob
import string
import zarr
import warnings
from tqdm.notebook import tqdm
from scipy import stats as sstats

import matplotlib as mpl
from matplotlib import pyplot as plt

from funcs_support import get_filepaths,get_params,utility_save,utility_print,load_shp,nan_argmax_xr
from funcs_readregs import coeffs_mortality, coeffs_gdppc, coeffs_payroll

dir_list = get_params()



In [3]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':False},
               'gdppc':{'varname':'tasmeanpoly','geo_typ':'adm1','popwght':True},
               'payroll':{'varname':'days90plus','geo_typ':'adm2-us','popwght':True}}

nsamples = 1000
rng = np.random.default_rng()

In [4]:
def get_coeff_draws(proc,load_kwargs = {},
                    rng = rng,
                    nsamples = nsamples):
    ''' Get MC draws of coefficients

    Parameters
    ---------------
    proc : str
        Name of impact variable

    load_kwargs : dict
        Piped into `coeffs_*()`

    rng : numpy random generator

    nsamples : int
        Number of MC samples of VCV to use

    Returns
    ---------------
    coeffs_mvn : xr.DataArray
    '''

    # Load regression coefficients
    if proc == 'gdppc':
        coeffs,vcv = coeffs_gdppc(**load_kwargs)
    elif proc == 'mortality':
        coeffs,vcv = coeffs_mortality(**load_kwargs)
    elif proc == 'payroll':
        coeffs,vcv = coeffs_payroll(**load_kwargs)
    else:
        raise KeyError()

    coeffs = coeffs.dropna('proj_base',how='all')
    if 'proj_base_1' in vcv:
        vcv.dropna('proj_base_1').dropna('proj_base_2')
    else:
        vcv = vcv.dropna('proj_base',how='all')

    # Get MC samples of coefficients
    if 'proj_base_1' in vcv:
        # If jointly estimated across parameters
        coeffs_tmp = coeffs.coeffs.stack(predproj = ['pred','proj_base'])
        vcv_tmp = vcv.vcv.stack(predproj_1 = ['pred_1','proj_base_1'],
                                 predproj_2 = ['pred_2','proj_base_2'])
        
        coeffs_mvn = rng.multivariate_normal(coeffs_tmp,
                                             vcv_tmp,
                                             size=nsamples)
        
        coeffs_mvn = xr.DataArray(coeffs_mvn,
                                  dims = ['sidx','predproj'],
                                  coords = {'predproj':coeffs_tmp.predproj})
        coeffs_mvn = coeffs_mvn.unstack('predproj')
    else:
        # If separate fit for each proj_base
        coeffs_mvn = xr.apply_ufunc(rng.multivariate_normal,
                                   coeffs.coeffs,
                                   vcv.vcv,
                                   input_core_dims = [['pred'],['pred_1','pred_2']],
                                   output_core_dims = [['sidx','pred']],
                                   kwargs = {'size':nsamples},
                                           dask = 'parallelized',
                                           dask_gufunc_kwargs = {'output_sizes':{'sidx':nsamples}},
                                   vectorize=True)
    return coeffs_mvn

In [5]:
def load_dmgfs(proc,load_kwargs = {},
              rng = rng,
              nsamples = nsamples):
    ''' Load MC'ed damage function, sampled at temperatures Tref

    Parameters
    ---------------
    proc : str
        Name of impact variable

    load_kwargs : dict
        Piped into `coeffs_*()`

    rng : numpy random generator

    nsamples : int
        Number of MC samples of VCV to use

    Returns
    ---------------
    dmgf : xr.DataArray
        `Tref` x `proj_base` x `sidx` datarray giving 
        coefficients at temperatures `Tref`


    '''
    
    # Load regression coefficients
    if proc == 'gdppc':
        coeffs,vcv = coeffs_gdppc(**load_kwargs)
    elif proc == 'mortality':
        coeffs,vcv = coeffs_mortality(**load_kwargs)
    elif proc == 'payroll':
        coeffs,vcv = coeffs_payroll(**load_kwargs)
    else:
        raise KeyError()

    coeffs = coeffs.dropna('proj_base',how='all')
    if 'proj_base_1' in vcv:
        vcv.dropna('proj_base_1').dropna('proj_base_2')
    else:
        vcv = vcv.dropna('proj_base',how='all')

    # Get MC samples of coefficients
    if 'proj_base_1' in vcv:
        # If jointly estimated across parameters
        coeffs_tmp = coeffs.coeffs.stack(predproj = ['pred','proj_base'])
        vcv_tmp = vcv.vcv.stack(predproj_1 = ['pred_1','proj_base_1'],
                                 predproj_2 = ['pred_2','proj_base_2'])
        
        coeffs_mvn = rng.multivariate_normal(coeffs_tmp,
                                             vcv_tmp,
                                             size=nsamples)
        
        coeffs_mvn = xr.DataArray(coeffs_mvn,
                                  dims = ['sidx','predproj'],
                                  coords = {'predproj':coeffs_tmp.predproj})
        coeffs_mvn = coeffs_mvn.unstack('predproj')
    else:
        # If separate fit for each proj_base
        coeffs_mvn = xr.apply_ufunc(rng.multivariate_normal,
                                   coeffs.coeffs,
                                   vcv.vcv,
                                   input_core_dims = [['pred'],['pred_1','pred_2']],
                                   output_core_dims = [['sidx','pred']],
                                   kwargs = {'size':nsamples},
                                           dask = 'parallelized',
                                           dask_gufunc_kwargs = {'output_sizes':{'sidx':nsamples}},
                                   vectorize=True)

    if np.any(['sumpoly' in k for k in coeffs_mvn.pred.values]):
        ts = np.arange(-20,51)
        var_stem = 'tassumpoly_'
        poly_degs = [int(re.split(var_stem,v)[-1]) for v in coeffs_mvn.pred.values if var_stem in v]
        t_vals = xr.DataArray(np.vstack([ts**k for k in poly_degs]),
                     dims = ['pred','bin_c'],
                     coords = {'pred':[v for v in coeffs_mvn.pred.values if var_stem in v],
                               'bin_c':ts})
    
        # Get damage function
        dmgf = (coeffs_mvn*t_vals).sum('pred')
        
        if proc == 'mortality':
            # Normalize over lowest value
            dmgf = dmgf - dmgf.sel(bin_c = slice(10,30)).min('bin_c')
        elif proc == 'gdppc':
            # Normalize over highest value
            dmgf = dmgf - dmgf.sel(bin_c = slice(10,30)).max('bin_c')
            #dmgf = dmgf - dmgf.sel(bin_c = 13)
        
        dmgf = dmgf.rename({'bin_c':'Tref'})
    elif 'bin' in coeffs_mvn.pred.values[0]:
        # Constructing the single-bin version
        dmgf = xr.apply_ufunc(rng.normal,
                   coeffs.isel(pred=0),
                   vcv.vcv.isel(pred_1 = 0,pred_2 = 0),
                   output_core_dims = [['sidx']],
                   vectorize=True,
                   kwargs = {'size':nsamples})
        
        dmgf = dmgf.expand_dims({'bin_f':np.arange(float(re.split(r'tasmaxbinF\_',str(coeffs.pred.values[0]))[-1]),
                                             125,5)})
        
        dmgf = xr.concat([xr.zeros_like(dmgf.isel(bin_f=0,drop=True)).expand_dims({'bin_f':np.arange(float(re.split(r'tasmaxbinF\_',str(coeffs.pred.values[0]))[-1])-20,
                                                                                   float(re.split(r'tasmaxbinF\_',str(coeffs.pred.values[0]))[-1]),
                                                                                   5)}),
                          dmgf],
                         dim='bin_f')
    
        dmgf = dmgf.rename({'bin_f':'Tref'}).coeffs
    elif 'days' in coeffs_mvn.pred.values[0]:
        dmgf = coeffs_mvn.expand_dims({'bin_f':np.arange(92.5, 125,5)}).isel(pred=0)
        dmgf = xr.concat([xr.zeros_like(dmgf.isel(bin_f=0,drop=True)).expand_dims({'bin_f':np.arange(92.5-20,
                                                                                           92.5,
                                                                                           5)}),
                                  dmgf],
                                 dim='bin_f')
        dmgf = dmgf.rename({'bin_f':'Tref'})

    return dmgf

### Calendar years for each GWL
At first mention, GWL 2 ("reached on average in year XXXX across the BCD-ME")

In [6]:
# Get paths of BCD-ME files
bcd_fns = pd.DataFrame({'path':np.sort(glob.glob(dir_list['bcd_me']+'*.zarr'))})
bcd_fns['model'] = [re.split(r'\.',re.split(r'\_',fn)[-1])[0] for fn in bcd_fns.path]
bcd_fns['typ'] = [re.split(r'_',re.split(r'\/',fn)[-1])[2] for fn in bcd_fns.path.values]
modlist = bcd_fns['model'].unique()
bcd_fns = bcd_fns.set_index(['typ','model'])
bcd_fns = bcd_fns.loc['qdm']

In [7]:
with warnings.catch_warnings():
    #.done is not recognized flag
    warnings.filterwarnings('ignore')
    calendar_years = xr.concat([xr.open_zarr(fn,consolidated=False).calendar_year
                                for fn in bcd_fns.path.values],
                               dim='idv')

In [8]:
print(f'Average year to pass GWL2, full ensemble: {str(int(calendar_years.sel(gwl=2).mean('year').mean('idv').load().values))}')
for ssp in ['ssp245','ssp370','ssp585']:
    print(f'Average year to pass GWL2, {ssp} runs only: {str(int(calendar_years.sel(gwl=2,experiment=ssp).mean('year').mean('idv').load().values))}')



Average year to pass GWL2, full ensemble: 2040
Average year to pass GWL2, ssp245 runs only: 2041
Average year to pass GWL2, ssp370 runs only: 2039
Average year to pass GWL2, ssp585 runs only: 2038


### The inflection point ("ideal temperature") for Burke et al. GDP per capita growth

In [9]:
dmgf = load_dmgfs('gdppc')

In [10]:
infl_points = dmgf.Tref.isel(Tref = dmgf.mean('sidx').argmax('Tref'))

for pb in infl_points.proj_base.values:
    print(f'{pb}: {str(infl_points.sel(proj_base=pb).values)} degC')

ERA5: 13 degC
GMFD: 16 degC
JRA-3Q: 12 degC
MERRA2: 16 degC
UDel: 14 degC


### Proporton of regression uncertainty as largest source by area, population

In [13]:
gwl = 2
for proc in params_proc:

    # Load shapefile# Load shapefile
    gdf = load_shp(proc)
    
    # Load partitioned uncertainty 
    fn_uncerts = f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}_wregunc.zarr'
    uncerts_all = xr.open_zarr(fn_uncerts).isel(sample=0)
    uncerts_all = uncerts_all.set_index({'poly_idx':'locid'})
    plot_data = uncerts_all.sel(gwl=gwl,typ='norm',poly_idx=gdf.locid.values).load()
    
    # Get which uncertainty source is most important 
    srcs = plot_data.to_dataarray()
    src_idxs = nan_argmax_xr(srcs,dim='variable').dropna('poly_idx').astype(int)
    srcs = srcs.sel(poly_idx = src_idxs.poly_idx).isel(variable=src_idxs)['variable']
    srcs = srcs.to_dataframe()['variable'].iloc[:,[0]]
    
    srcs = srcs.reset_index().rename({'poly_idx':'locid'},axis=1)

    # Merge into gdf
    gdf = gdf.set_index('locid').join(srcs.set_index('locid'))
    gdf = gdf.iloc[(~gdf['variable'].isnull()).values,:]
    gdf = gdf.join(plot_data['uncreg'].rename({'poly_idx':'locid'}).to_dataframe()['uncreg'])

    # Reproj into WGS84 for area calcs
    gdf = gdf.to_crs('EPSG:4087')

    print('\n---- '+proc+':')

    # Get relative area of globe for each
    gdf_areas = gdf.dissolve(by='variable').area
    print('\n-- by area')
    print(str(np.round((gdf.uncreg*gdf.area).sum()/gdf.area.sum(),2)))
    print(str(gdf_areas / gdf_areas.sum()))

    # Now, get by population
    if proc == 'mortality':
        preds = xr.open_zarr(dir_list['proc']+'aux/all_preds.zarr')
        pop = preds[['pop']].sel(gwl=0.61,ageset='agetotal',pred_ssp='ssp3',pred_model='high')
        pop = pop.mean('year').load()

        gdf = gdf.join(pop.rename({'hierid':'locid'}).to_dataframe()['pop'])

        gdf_pops = gdf.groupby('variable')['pop'].sum()
    elif proc == 'gdppc':
        gdf_pops = gdf.groupby('variable')['POP_CNTRY'].sum() 
        gdf = gdf.rename({'POP_CNTRY':'pop'},axis=1)
    elif proc == 'payroll':
        # Load population by grid cell
        pop_fn = dir_list['aux']+'pop_data/pop_fx_LANDSCAN_conus_day_025deg_2016.nc'
        pop = xr.open_dataset(pop_fn)
    
        with xa.set_options(silent=True):
            wm = xa.pixel_overlaps(pop,gdf.to_crs('EPSG:4326'))
            with xa.set_options(impl='numba'):
                agg = xa.aggregate(pop,wm)
                agg_out = agg.to_dataset()
        
        agg_out['locid'] = agg_out['STATE_FIPS']+agg_out['CNTY_FIPS']
        gdf = gdf.join(agg_out.to_dataframe().set_index('locid')['pop'])
        gdf['pop'] = (gdf.to_crs('EPSG:4087').geometry.area*gdf['pop'])
        gdf_pops = gdf.groupby('variable')['pop'].sum()
    print('\n-- by pop')
    print(str(np.round((gdf['uncreg']*gdf['pop']).sum()/gdf['pop'].sum(),2)))
    print(str(gdf_pops / gdf_pops.sum()))


---- mortality:

-- by area
0.4
variable
uncmod    0.016146
uncobs    0.640121
uncreg    0.343733
dtype: float64

-- by pop
0.43
variable
uncmod    0.120986
uncobs    0.308565
uncreg    0.570449
Name: pop, dtype: float64

---- gdppc:

-- by area
0.63
variable
uncmod    0.050044
uncobs    0.000035
uncreg    0.949921
dtype: float64

-- by pop
0.58
variable
uncmod    0.021794
uncobs    0.000179
uncreg    0.978027
Name: POP_CNTRY, dtype: float64

---- payroll:

-- by area
0.34
variable
uncint    0.028374
uncmod    0.001893
uncobs    0.593096
uncreg    0.376637
dtype: float64

-- by pop
0.36
variable
uncint    0.015039
uncmod    0.000248
uncobs    0.520256
uncreg    0.464457
Name: pop, dtype: float64


### Proportion of observational uncertainty as largest source by area, population

In [12]:
imp_params = {'mortality':{'timeframe':'1980-2010','obstreat':'sep','resolution':'025deg','spec':'pooled','indflag':'noInd'},
              'gdppc':{'timeframe':'1980-2010','obstreat':'sep','resolution':'025deg'},
              'payroll':{'timeframe':'1986-2008','obstreat':'sep','resolution':'025deg','weighting':'pop'}}

for proc in imp_params:
    samples = {'bygwl':'13runs8mods',
           'byyr':'atleast3runs'}
    if proc == 'payroll':
        samples = {'bygwl':'10runs7mods',
           'byyr':'atleast3runs'}
    
    fn_uncerts = {'bygwl':f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}.zarr',
                  'byyr':f'{dir_list['proc']}uncert_parts/uncert_parts_{proc}_byyr.zarr'}
    uncerts_all = {typ:xr.open_zarr(fn_uncerts[typ]).sel(**imp_params[proc],sample=samples[typ]) for typ in fn_uncerts}
    uncerts_all = {typ:d.to_dataarray().sel(typ='norm') for typ,d in uncerts_all.items()}
    
    print('\n---------- '+proc+':')
    for typ in uncerts_all:
        
        if typ == 'bygwl':
            gwl_list = [1,1.5,2,2.5,3]
        elif typ == 'byyr':
            gwl_list = [2010,2040,2070]
    
        
        # Get which uncertainty source is most important 
        srcs = uncerts_all[typ].load()
        src_idxs = nan_argmax_xr(srcs,dim='variable').dropna('poly_idx',how='all').dropna('gwl',how='all').astype(int)
        src_idxs = src_idxs.where(src_idxs>=0).dropna('poly_idx').astype(int)
        srcs = srcs.sel(poly_idx = src_idxs.where(src_idxs>=0).poly_idx,gwl=src_idxs.gwl.values).isel(variable=src_idxs)['variable']
        srcs = srcs.set_index(poly_idx = 'locid').rename({'poly_idx':'locid'})
        srcs = srcs.drop_vars(['model','proj_base','numruns','variable',
                               *[k for k in imp_params[proc]]]).to_dataframe()#['variable']#.iloc[:,[0]]
        srcs = srcs.reset_index().set_index(['gwl','locid']).sort_index()[['variable']]
    
        # Load shapefiles
        gdf = load_shp(proc)
        # Merge into gdf
        gdf = gdf.set_index('locid').join(srcs)
        gdf = gdf.iloc[(~gdf['variable'].isnull()).values,:]
        gdf = gdf.join((uncerts_all[typ].set_index(poly_idx='locid').rename({'poly_idx':'locid'}).sel(variable='uncobs').
         drop_vars(['model','proj_base','numruns','variable',
                               *[k for k in imp_params[proc]]]).to_dataframe(name='uncobs'))[['uncobs']])
        
        
        # Reproj into WGS84 for area calcs
        gdf = gdf.to_crs('EPSG:4087')
        
        print('\n---- '+typ+':')
        
        # Get relative area of globe for each
        gdf_areas = gdf.dissolve(by=['variable','gwl']).area
        gdf_areas = gdf_areas.to_frame().reset_index().set_index('gwl').pivot(columns=['variable'])
        gdf_areas = gdf_areas.loc[gwl_list,:]
        print('\n-- by area')
        print(gdf_areas / gdf_areas.sum(axis=1).to_frame())
        
        # Now, get by population
        if proc == 'mortality':
            preds = xr.open_zarr(dir_list['proc']+'aux/all_preds.zarr')
            pop = preds[['pop']].sel(gwl=0.61,ageset='agetotal',pred_ssp='ssp3',pred_model='high')
            pop = pop.mean('year').load()
        
            gdf = gdf.join(pop.rename({'hierid':'locid'}).to_dataframe()['pop'])
        
            gdf_pops = gdf.groupby(['variable','gwl'])['pop'].sum()
        elif proc == 'gdppc':
            gdf_pops = gdf.groupby(['variable','gwl'])['POP_CNTRY'].sum() 
            gdf = gdf.rename({'POP_CNTRY':'pop'},axis=1)
        elif proc == 'payroll':
            # Load population by grid cell
            pop_fn = dir_list['aux']+'pop_data/pop_fx_LANDSCAN_conus_day_025deg_2016.nc'
            pop = xr.open_dataset(pop_fn)
        
            with xa.set_options(silent=True):
                wm = xa.pixel_overlaps(pop,gdf.to_crs('EPSG:4326'))
                with xa.set_options(impl='numba'):
                    agg = xa.aggregate(pop,wm)
                    agg_out = agg.to_dataset()
            
            agg_out['locid'] = agg_out['STATE_FIPS']+agg_out['CNTY_FIPS']
            agg_out = agg_out.to_dataframe().set_index('locid')
            agg_out['gwl'] = gdf.reset_index()['gwl'].values
            agg_out = agg_out.reset_index().set_index(['gwl','locid'])
            gdf = gdf.join(agg_out[['pop']])
            gdf['pop'] = (gdf.to_crs('EPSG:4087').geometry.area*gdf['pop'])
            gdf_pops = gdf.groupby(['variable','gwl'])['pop'].sum()
        
        gdf_pops = gdf_pops.to_frame().reset_index().set_index('gwl').pivot(columns=['variable'])
        
        gdf_pops = gdf_pops.loc[gwl_list,:]
        print('\n-- by pop')
        print(gdf_pops.div(gdf_pops.sum(axis=1), axis=0))


---------- mortality:

---- bygwl:

-- by area
                 0                    
variable    uncint    uncmod    uncobs
gwl                                   
1.0       0.332600  0.012059  0.655341
1.5       0.090049  0.044542  0.865409
2.0       0.013535  0.056265  0.930200
2.5       0.001668  0.048222  0.950110
3.0       0.000161  0.044905  0.954934

-- by pop
               pop                    
variable    uncint    uncmod    uncobs
gwl                                   
1.0       0.689583  0.016603  0.293814
1.5       0.132258  0.198832  0.668910
2.0       0.006319  0.210182  0.783499
2.5       0.000130  0.141477  0.858393
3.0       0.000002  0.091700  0.908298

---- byyr:

-- by area
                 0                    
variable    uncint    uncmod    uncobs
gwl                                   
2010.0    0.208083  0.132082  0.659836
2040.0    0.022449  0.114201  0.863350
2070.0    0.000098  0.145383  0.854519

-- by pop
               pop                    
variable 

/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/duck_array_ops.py:268: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)



-- by area
                 0                    
variable    uncint    uncmod    uncobs
gwl                                   
1.0       0.698196  0.002280  0.299523
1.5       0.210929  0.013122  0.775949
2.0       0.040541  0.003557  0.955902
2.5       0.009594  0.007049  0.983358
3.0       0.003179  0.018674  0.978147

-- by pop
               pop                    
variable    uncint    uncmod    uncobs
gwl                                   
1.0       0.583117  0.012307  0.404577
1.5       0.227803  0.020896  0.751302
2.0       0.080348  0.001692  0.917960
2.5       0.040275  0.002446  0.957279
3.0       0.001531  0.026247  0.972221

---- byyr:


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/xarray/core/duck_array_ops.py:268: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)



-- by area
                 0                    
variable    uncint    uncmod    uncobs
gwl                                   
2010.0    0.595229  0.009060  0.395710
2040.0    0.009084  0.106033  0.884883
2070.0         NaN  0.099231  0.900769

-- by pop
               pop                    
variable    uncint    uncmod    uncobs
gwl                                   
2010.0    0.525475  0.004806  0.469719
2040.0    0.001426  0.122569  0.876005
2070.0         NaN  0.105954  0.894046


## Reanalysis differences

In [5]:
from distributed import Client
client = Client()
display(client)

/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33893 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/33893/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/33893/status,Workers: 2
Total threads: 2,Total memory: 10.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38299,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/33893/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:45955,Total threads: 1
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/42663/status,Memory: 5.00 GiB
Nanny: tcp://127.0.0.1:37273,


2026-06-11 12:06:33,284 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 7f15cb4d2dcfdbc68a771b8c1161aaf8 initialized by task ('rechunk-merge-rechunk-transfer-6cb1c4316e7274db5fd03cb5fed01e12', 0, 6, 0, 0, 6, 0) executed on worker tcp://127.0.0.1:36717
2026-06-11 12:06:33,293 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle e9fcd622cdbbcaca4130317ebc31528d initialized by task ('rechunk-merge-rechunk-transfer-6cb1c4316e7274db5fd03cb5fed01e12', 1, 6, 0, 1, 6, 0) executed on worker tcp://127.0.0.1:45955
2026-06-11 12:06:33,319 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 17a15fbb0884a23e5d1aec43af7f102b initialized by task ('rechunk-merge-rechunk-transfer-6cb1c4316e7274db5fd03cb5fed01e12', 2, 6, 0, 2, 6, 0) executed on worker tcp://127.0.0.1:45955
2026-06-11 12:06:33,597 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 7f15cb4d2dcfdbc68a771b8c1161aaf8 deactivated due to stimulus 'task-finished-1781201193.5945892'
2026-06-11 12:06:33,625 - di

In [6]:
from funcs_aux import _restore_doys
import xesmf as xe
dss = []
for mod in ['ERA5-025','MERRA2','JRA-3Q','GMFD']:
    ds = xr.open_zarr(f'/glade/derecho/scratch/schwarzwald/climate_proc/{mod}/tas_day_{mod}_historical_reanalysis_CACHE-RESHAPE.zarr')
    ds = _restore_doys(ds,{'wwidth':31})
    ds = ds.expand_dims({'model':[mod]})
    dss.append(ds)
dss = xr.concat(dss,dim='model')

In [7]:
pop_fn = f'{dir_list['aux']}pop_data/gpw_v4_population_count_rev11_025deg_rg.nc'
pop = xr.open_dataset(pop_fn)
rgrd = xe.Regridder(pop,dss,'conservative')
pop = rgrd(pop)

In [8]:
dss_metrics = xr.merge([dss.mean(('dayofyear','year')).rename({'tas':'mu'}),
                        dss.quantile(0.95,('dayofyear','year')).rename({'tas':'q95'}).drop_vars('quantile'),
                        dss.quantile(0.99,('dayofyear','year')).rename({'tas':'q99'}).drop_vars('quantile'),
                        dss.quantile(0.999,('dayofyear','year')).rename({'tas':'q999'}).drop_vars('quantile')])
dss_metrics_spread = xr.concat([dss_metrics.std('model'),
                                np.abs(dss_metrics.max('model') - dss_metrics.min('model')),
                                np.abs(dss_metrics.sel(model='ERA5-025') - dss_metrics.sel(model='MERRA2'))],
                               dim=pd.Index(['std','range','era5-merra2'],name='typ'))
dss_metrics_spread_popw = (dss_metrics_spread*pop.pop).sum(('lat','lon')) / pop.pop.sum(('lat','lon'))
dss_metrics_spread_popw = dss_metrics_spread_popw.compute()


In [9]:
for var in dss_metrics_spread_popw:
    print(f'\n{var}\n-------')
    print(f'Std. Dev. across reanalyses: {str(np.round(dss_metrics_spread_popw[var].sel(typ='std').values,2))}')
    print(f'Range across reanalyses: {str(np.round(dss_metrics_spread_popw[var].sel(typ='range').values,2))}')
    print(f'ERA5-MERRA2: {str(np.round(dss_metrics_spread_popw[var].sel(typ='era5-merra2').values,2))}')
    


mu
-------
Std. Dev. across reanalyses: 0.46
Range across reanalyses: 1.18
ERA5-MERRA2: 0.71

q95
-------
Std. Dev. across reanalyses: 0.67
Range across reanalyses: 1.73
ERA5-MERRA2: 1.09

q99
-------
Std. Dev. across reanalyses: 0.78
Range across reanalyses: 2.03
ERA5-MERRA2: 1.17

q999
-------
Std. Dev. across reanalyses: 0.88
Range across reanalyses: 2.28
ERA5-MERRA2: 1.26


In [12]:
gdf = load_shp('gdppc')
import xagg as xa
wm = xa.pixel_overlaps(dss_metrics_spread,gdf,weights=pop.pop)
with xa.set_options(impl='numba'):
    agg = xa.aggregate(dss_metrics_spread.chunk({'lat':-1,'lon':-1}),wm)

creating polygons for each pixel...
calculating overlaps between pixels and output polygons...
success!
aggregating mu...


/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 13.88 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


aggregating q95...


/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 13.91 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


aggregating q99...


/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 13.91 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


aggregating q999...


/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 13.91 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


aggregating weights...
all variables aggregated to polygons!


/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 13.78 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [20]:
ds_agg = agg.to_dataset()
ds_agg = ds_agg.set_coords(['FIPS_CNTRY','GMI_CNTRY','locid','SOVEREIGN','POP_CNTRY',
                   'SQKM_CNTRY','SQMI_CNTRY','CURR_TYPE','CURR_CODE','LANDLOCKED',
                   'COLOR_MAP','fid2','changefrom']).drop_vars('weights').set_index(poly_idx='locid')

In [54]:
for loc in ['Pakistan','Guatemala']:
    print('---------\nFor '+loc+':')
    for var in ['mu','q99']:
        print(var+': '+str(np.round(ds_agg.sel(typ='era5-merra2',poly_idx=loc)[var].values,2)))

---------
For Pakistan:
mu: 1.47
q99: 2.35
---------
For Guatemala:
mu: 1.88
q99: 2.88


## Table of models used

In [6]:
def ssp_parser(name):
    ''' Convert from lowercase filename SSPs to "official" SSP names '''
    m = re.match(r'ssp(\d)(\d)(\d)(-over)?', name.lower())
    if not m:
        raise ValueError(f'Unrecognized SSP format: {name}')

    major, tens, ones, over = m.groups()
    forcing = f'{tens}.{ones}'
    suffix = '-OS' if over else ''

    return f'SSP{major}-{forcing}{suffix}'

In [7]:
fns = glob.glob(dir_list['bcd_me']+'*_qdm_*.zarr')
fns = np.sort(fns)

has_data_array = xr.concat([xr.open_zarr(fn).has_data for fn in fns],
          dim='idv',join='outer').load()
has_data_array = has_data_array.where(~np.isnan(has_data_array),0)

/glade/derecho/scratch/schwarzwald/tmp/ipykernel_58626/2347832078.py:4: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  has_data_array = xr.concat([xr.open_zarr(fn).has_data for fn in fns],
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/group.py:3289: ZarrUserWarning: Object at .done_v021 is not recognized as a component of a Zarr hierarchy.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/core/group.py:3289: ZarrUserWar

In [8]:
total_nruns = has_data_array.any('gwl').sum('idv')
print(f'Number of total runs (`tas`): {str(total_nruns.sel(variable='tas').values)}')
print(f'Number of total runs (`tasmax`): {str(total_nruns.sel(variable='tasmax').values)}')
print(f'Number of models: {str(len(fns))}')

Number of total runs (`tas`): 362
Number of total runs (`tasmax`): 199
Number of models: 22


In [9]:
df_bygwl = has_data_array.sel(gwl=slice(1,None)).set_index(idv = ['model','run','experiment']).unstack().sum(('run','experiment')).rename({'gwl':'GWL reached:'}).to_dataframe().reset_index().pivot(columns = 'GWL reached:',
                                                                                                                      index = ['variable','model'])
df_bygwl.columns = df_bygwl.columns.droplevel()

# Get total number of runs by GWL
totals = df_bygwl.groupby(['variable']).sum().astype(int)
totals['model'] = 'Total'
totals = totals.reset_index().set_index(['variable','model'])
# Concat back in
df_bygwl = pd.concat([df_bygwl,totals]).sort_index()
# Drop models without data
df_bygwl = df_bygwl.where(~(df_bygwl==0).all(axis=1)).dropna()

# Capitalize
df_bygwl.index = df_bygwl.index.rename({'model':'Model','variable':'Variable'})


# Change to integer dtype
df_bygwl = df_bygwl.astype(int)

# Print latex table
print(df_bygwl.to_latex())



\begin{tabular}{llrrrrrrr}
\toprule
 & GWL reached: & 1.000000 & 1.500000 & 2.000000 & 2.500000 & 3.000000 & 3.500000 & 4.000000 \\
Variable & Model &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{23}{*}{tas} & ACCESS-CM2 & 1 & 1 & 1 & 1 & 1 & 1 & 0 \\
 & ACCESS-ESM1-5 & 24 & 24 & 24 & 24 & 5 & 5 & 5 \\
 & BCC-CSM2-MR & 2 & 2 & 2 & 2 & 2 & 1 & 0 \\
 & BCC-ESM1 & 1 & 1 & 0 & 0 & 0 & 0 & 0 \\
 & CAMS-CSM1-0 & 1 & 1 & 1 & 1 & 0 & 0 & 0 \\
 & CESM2-WACCM & 8 & 8 & 8 & 7 & 7 & 4 & 3 \\
 & CMCC-CM2-SR5 & 1 & 1 & 1 & 1 & 1 & 1 & 1 \\
 & CMCC-ESM2 & 1 & 1 & 1 & 1 & 1 & 1 & 1 \\
 & CNRM-CM6-1 & 18 & 18 & 18 & 18 & 15 & 12 & 6 \\
 & CNRM-ESM2-1 & 41 & 41 & 31 & 26 & 13 & 7 & 5 \\
 & CanESM5 & 86 & 86 & 86 & 86 & 86 & 86 & 79 \\
 & EC-Earth3 & 35 & 35 & 35 & 35 & 32 & 15 & 10 \\
 & EC-Earth3-Veg & 13 & 13 & 13 & 13 & 13 & 9 & 6 \\
 & FGOALS-g3 & 10 & 10 & 10 & 7 & 3 & 0 & 0 \\
 & IITM-ESM & 1 & 1 & 1 & 1 & 1 & 0 & 0 \\
 & IPSL-CM6A-LR & 14 & 14 & 14 & 14 & 14 & 11 & 10 \\
 & MIROC-ES2L & 1 & 1 & 1 

In [12]:
# Save as csv for excel --> word pipeline
df_bygwl.to_csv(dir_list['aux']+'df_bygwl.csv')

In [67]:
df_long = has_data_array.any('gwl').groupby(['model','experiment']).sum().sum('variable').stack(allv = ['model','experiment'])
df_long = df_long.where(df_long>0,drop=True)
df_long = df_long.to_dataframe()
#df_long.columns = df_long.columns.droplevel()

# This will be used to create natbib file
cite_strs = dict()

# Add \citet{} strings to table
for row in df_long.iterrows():
    if row[0][0] not in cite_strs:
        cite_strs[row[0][0]] = dict()

    cite_code = '_'.join(row[0])
    df_long.loc[row[0],'cite'] = r'\citet{'+cite_code+r'}'
    cite_strs[row[0][0]][row[0][1]] = cite_code

df_long = df_long.drop('has_data',axis=1).reset_index().set_index('model')
mods = np.unique(df_long.index.values)
mod_cites = pd.DataFrame([','.join([ssp_parser(row[1]['experiment'])+row[1]['cite']
          for row in a.loc[[mod]].iterrows()])
              for mod in mods],
            index=mods,columns=['Reference'])


In [68]:
mod_cites

,Reference
ACCESS-CM2,"SSP3-7.0\citet{ACCESS-CM2_ssp370},SSP5-8.5\cit..."
ACCESS-ESM1-5,"SSP2-4.5\citet{ACCESS-ESM1-5_ssp245},SSP5-8.5\..."
BCC-CSM2-MR,"SSP3-7.0\citet{BCC-CSM2-MR_ssp370},SSP5-8.5\ci..."
BCC-ESM1,SSP3-7.0\citet{BCC-ESM1_ssp370}
CAMS-CSM1-0,SSP5-8.5\citet{CAMS-CSM1-0_ssp585}
CESM2-WACCM,"SSP2-4.5\citet{CESM2-WACCM_ssp245},SSP3-7.0\ci..."
CMCC-CM2-SR5,SSP5-8.5\citet{CMCC-CM2-SR5_ssp585}
CMCC-ESM2,SSP5-8.5\citet{CMCC-ESM2_ssp585}
CNRM-CM6-1,"SSP2-4.5\citet{CNRM-CM6-1_ssp245},SSP3-7.0\cit..."
CNRM-ESM2-1,"SSP1-1.9\citet{CNRM-ESM2-1_ssp119},SSP1-2.6\ci..."


In [78]:
mod_cites.loc[df_bygwl.index.get_level_values('Model')]

KeyError: "['Total'] not in index"